In [8]:
!mkdir -p data
!curl -L https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz \
     -o data/aclImdb_v1.tar.gz
!tar -xzf data/aclImdb_v1.tar.gz -C data


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 80.2M  100 80.2M    0     0   739k      0  0:01:51  0:01:51 --:--:--  759k


讀檔 + 建 Dataset（從資料夾讀 pos/neg）

In [6]:
from pathlib import Path
from torch.utils.data import Dataset
class FolderDataset(Dataset):
    def __init__(self, root_dir, split):
        root_dir = Path(root_dir)
        base = root_dir/ split
        self.samples = []
        for label_name, label in [("pos", 1), ("neg", 0)]:
            folder = base / label_name
            for fp in folder.glob("*.txt"):
                self.samples.append((fp, label))
    def __len__(self):
        return len(self.samples)
    def __getitem__(self,index):
        value,label = self.samples[index]
        text = value.read_text()
        return text,label
train_ds = FolderDataset("data/aclImdb","train")
test_ds = FolderDataset("data/aclImdb","test")
print(len(train_ds))    
print(len(test_ds))

25000
25000


Tokenizer（把文字變成「數字編號」）+ 建 vocab（這些編號的「查表」）

In [7]:
import re #正規表示式（Regular Expression）
from collections import Counter
token_re = re.compile(r"[A-Za-z']+")
def tokenlizer(string):
    return token_re.findall(string.lower())
counter = Counter()
for text,_ in train_ds:
    counter.update(tokenlizer(text))
itos = ["<pad>","<unk>"]   #index to string <pad>（padding）用來補齊句子長度,<unk>uuknown
stoi = {"<pad>":0,"<unk>":1}
index = 2
for word,freq in counter.most_common():
    if freq>=2:
        itos.append(word)
        stoi[word] = index 
        index+=1
    else:
        break
    if index > 20002:
        break
print(itos[4])


a


collate_fn（把變長序列 padding 成同長）+ DataLoader(讀入網路的整理)

In [8]:
import torch
from torch.utils.data import DataLoader

def numericalize(tokens, stoi, unk_idx):
    return [stoi.get(t, unk_idx) for t in tokens]#回傳token對應的index，找不到就回傳unk_idx

def collate_batch(batch):   #collate_fn = DataLoader「怎麼把一堆樣本湊成一個 batch」的規則
    texts, labels = zip(*batch) #拆開文字和編號

    token_ids_list = []
    lengths = []
    for text in texts:
        ids = numericalize(tokenlizer(text), stoi, 1)  #unk_idx=1
        token_ids_list.append(torch.tensor(ids, dtype=torch.long))
        lengths.append(len(ids))

    lengths = torch.tensor(lengths, dtype=torch.long)

    # padding 到同長度 [B, T]
    padded = torch.nn.utils.rnn.pad_sequence(
        token_ids_list, batch_first=True, padding_value=0
    )

    # labels -> float [B,1] for BCEWithLogitsLoss
    y = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)

    return padded, lengths, y

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collate_batch) 
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, collate_fn=collate_batch)

x, lengths, y = next(iter(train_loader))
print("x:", x.shape, x.dtype)          # [B, T]
print("lengths:", lengths.shape)       # [B]
print("y:", y.shape, y.dtype)          # [B, 1]


x: torch.Size([32, 824]) torch.int64
lengths: torch.Size([32])
y: torch.Size([32, 1]) torch.float32


LSTM 模型（用 pack_padded_sequence 更標準）

In [9]:
import torch.nn as nn

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, num_layers=1, pad_idx=0, bidirectional=False):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
        )
        out_dim = hidden_dim * (2 if bidirectional else 1)
        self.fc = nn.Linear(out_dim, 1)

    def forward(self, x, lengths):
        # x: [B, T]
        emb = self.embedding(x)  # [B, T, E]

        packed = nn.utils.rnn.pack_padded_sequence(
            emb, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, (h_n, _) = self.lstm(packed)
        # h_n: [num_layers*(2 if bi else 1), B, H]
        last = h_n[-1]  # [B, H] (或 bi 的其中一個方向最後層；入門先這樣最清楚)

        logits = self.fc(last)  # [B, 1]
        return logits

device = "cuda" if torch.cuda.is_available() else "cpu"
model = LSTMClassifier(vocab_size=len(itos), embed_dim=128, hidden_dim=128, pad_idx=0).to(device)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print("device:", device)


device: cpu


先跑 1 個 epoch

In [ ]:
def accuracy_from_logits(logits, y):
    probs = torch.sigmoid(logits)
    preds = (probs >= 0.5).float()
    return (preds == y).float().mean().item()

model.train()
total_loss = 0.0
total_acc = 0.0
n_batches = 0

for x, lengths, y in train_loader:
    x, lengths, y = x.to(device), lengths.to(device), y.to(device)

    optimizer.zero_grad()
    logits = model(x, lengths)
    loss = loss_fn(logits, y)
    loss.backward()
    optimizer.step()

    total_loss += loss.item()
    total_acc += accuracy_from_logits(logits.detach(), y)
    n_batches += 1

    # 先跑少量 batch 也行，想快就把這行打開
    # if n_batches == 200: break

print("train_loss:", total_loss / n_batches)
print("train_acc:", total_acc / n_batches)


train_loss: 0.6410872488070631
train_acc: 0.6250399616368286


改用育訓練好的embadding100維的

In [10]:
# Alternative: load GloVe with a programmatic download (no torchtext needed)
import io, zipfile, urllib.request, os
import torch

glove_url = 'https://nlp.stanford.edu/data/glove.6B.zip'
target_dim = 100

# Download once into ./data if missing
zip_path = 'data/glove.6B.zip'
os.makedirs('data', exist_ok=True)
if not os.path.exists(zip_path):
    print('Downloading GloVe...')
    urllib.request.urlretrieve(glove_url, zip_path)


In [ ]:

# Read the specific dimension file from the zip
vectors = {}
with zipfile.ZipFile(zip_path) as zf:
    fname = f'glove.6B.{target_dim}d.txt'
    with zf.open(fname) as f:
        for line in io.TextIOWrapper(f, encoding='utf-8'): #是把「位元組 (bytes) 流」包成「文字 (str) 流」的工具。
            parts = line.rstrip().split(' ') #rstrip() 去掉尾端空白 split(' ') 以空白分割 先跑。line 的範例the 0.418 0.24968 -0.41242 0.1217 ... (共 target_dim 個數字)
            token = parts[0]
            vec = torch.tensor([float(x) for x in parts[1:]], dtype=torch.float32)
            vectors[token] = vec #vectors，key 是字，value 是對應的 glove 向量

# Build embedding matrix aligned with current vocab (itos)
weights = torch.randn(len(itos), target_dim) * 0.02 #隨機初始化 [num_embeddings, embedding_dim] 只用改用到的字
for i, token in enumerate(itos):   # 改現在用到的字，token 是字
    if token in vectors:
        weights[i] = vectors[token]
    elif token.lower() in vectors:
        weights[i] = vectors[token.lower()]

# New model for comparison (does NOT modify your existing model)
model_glove = LSTMClassifier(
    vocab_size=len(itos),
    embed_dim=100, # embadding維度
    hidden_dim=128,
    pad_idx=0
).to(device)

model_glove.embedding.weight.data.copy_(weights)
# Optional: freeze pretrained embeddings
# model_glove.embedding.weight.requires_grad = False

optimizer_glove = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model_glove.parameters()), lr=1e-3
)


In [ ]:
def accuracy_from_logits(logits, y):
    probs = torch.sigmoid(logits)
    preds = (probs >= 0.5).float()
    return (preds == y).float().mean().item()

model_glove.train()
total_loss = 0.0
total_acc = 0.0
n_batches = 0

for x, lengths, y in train_loader:
    x, lengths, y = x.to(device), lengths.to(device), y.to(device)

    optimizer_glove.zero_grad()
    logits = model_glove(x, lengths)
    loss = loss_fn(logits, y)
    loss.backward()
    optimizer_glove.step() #梯度更新參數

    total_loss += loss.item()
    total_acc += accuracy_from_logits(logits.detach(), y)
    n_batches += 1

    # 先跑少量 batch 也行，想快就把這行打開
    # if n_batches == 200: break

print("train_loss:", total_loss / n_batches)
print("train_acc:", total_acc / n_batches)


train_loss: 0.6512840170689556
train_acc: 0.628156969309463


In [ ]:
print(n_batches)
print(len(test_loader))
print(len(test_ds))
print(25000/32) #不滿直接跑小的

782
782
25000
781.25
